# 01: does the 3blue1brown network really learn edges?

**question:** if i train the exact network from 3blue1brown's videos (784 → 16 → 16 → 10), do the hidden neurons learn edges and loops like he hoped in chapter 1? or do they look like noise, like he says in chapter 2?

**what i found:** _fill in when done_

---

**how to use this notebook**
- run each cell in order with **shift + enter**
- most of the code is written for you. fill in every `___` (there are 9). each one connects to something in my notes
- if a cell errors, read the last line of the error first. it usually tells you what's wrong
- hints are in the comments. try without them first

## step 0: imports

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
import torchvision
import matplotlib.pyplot as plt

torch.manual_seed(0)  # makes results repeatable
print(torch.__version__)

## step 1: load the data
mnist = 60,000 training images + 10,000 test images of handwritten digits, each 28×28 pixels with a label (0 to 9).

`ToTensor()` turns each image into numbers between 0 and 1, exactly like the input-layer activations in chapter 1.

a `DataLoader` serves the images in **mini-batches** of 64 (the "stochastic" in stochastic gradient descent).

In [ ]:
to_tensor = torchvision.transforms.ToTensor()
train_data = torchvision.datasets.MNIST("data", train=True, download=True, transform=to_tensor)
test_data = torchvision.datasets.MNIST("data", train=False, download=True, transform=to_tensor)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=1000)

print(len(train_data), "training images,", len(test_data), "test images")

look at one image to check it loaded:

In [ ]:
image, label = train_data[0]
print("shape:", image.shape)  # 1 channel × 28 × 28
plt.imshow(image[0], cmap="gray")
plt.title(f"label: {label}")
plt.axis("off")
plt.show()

## step 2: build the network
from my notes: each layer computes $\mathbf{a}^{(1)} = \sigma\left(W\mathbf{a}^{(0)} + \mathbf{b}\right)$

in pytorch, `nn.Linear(n_in, n_out)` **is** $W\mathbf{a} + \mathbf{b}$: it holds a weight matrix $W$ (n_out × n_in) and a bias vector $\mathbf{b}$. `torch.sigmoid` is $\sigma$.

fill in the layer sizes and the second layer's activation.

In [ ]:
class TinyNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(784, 16)  # pixels -> hidden layer 1
        self.layer2 = nn.Linear(___, ___)  # hidden layer 1 -> hidden layer 2
        self.layer3 = nn.Linear(___, ___)  # hidden layer 2 -> 10 outputs (one per digit)

    def forward(self, x):
        a0 = x.view(-1, 784)                 # flatten each 28×28 image into 784 numbers
        a1 = torch.sigmoid(self.layer1(a0))  # σ(W a + b)
        a2 = ___                             # hint: same pattern as the line above, using layer2 and a1
        return self.layer3(a2)               # raw output scores (see step 3 for why there's no sigmoid here)

model = TinyNet()
print(model)

check: my notes say this network has **13,002** weights and biases. if this cell errors, recheck your layer sizes.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
print("parameters:", n_params)
assert n_params == 13002, "not 13,002. check the sizes in nn.Linear"
print("matches 3blue1brown!")

## step 3: the cost function and gradient descent
- **cost:** 3blue1brown uses squared error. here we use **cross-entropy**, the standard choice for classification. it's still "one number that says how bad the network is", it just trains faster. it applies its own squishing (softmax) to the outputs, which is why the last layer has no sigmoid
- **optimizer:** `SGD` does the update from my notes, $\mathbf{W} \leftarrow \mathbf{W} - \eta \, \nabla C$. `lr` is the learning rate $\eta$

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)

## step 4: train
one **epoch** = one pass over all 60,000 training images. for each mini-batch:
1. forward pass: compute the outputs and the cost
2. **backpropagation**: compute $\nabla C$
3. **gradient descent step**: nudge every weight and bias downhill

fill in the three key lines.

In [ ]:
def train_one_epoch():
    for images, labels in train_loader:
        outputs = model(images)           # forward pass
        loss = loss_fn(outputs, labels)   # cost C for this mini-batch

        optimizer.zero_grad()  # clear the gradient from the last mini-batch
        ___                    # backpropagation. hint: call .backward() on the cost
        ___                    # gradient descent step. hint: the optimizer has a .step() method
    return loss.item()


def test_accuracy():
    correct = 0
    with torch.no_grad():  # no gradients needed when just testing
        for images, labels in test_loader:
            guesses = model(images).argmax(dim=1)  # the brightest output neuron is the guess
            correct += (guesses == labels).sum().item()
    return correct / len(test_data)


for epoch in range(10):
    loss = train_one_epoch()
    print(f"epoch {epoch + 1}: cost {loss:.3f}, test accuracy {test_accuracy():.1%}")

**question:** how close did you get to 3blue1brown's 96%? _write your answer here_

## step 5: look inside (the mech interp part)
**hypothesis (chapter 1):** each of the 16 neurons in hidden layer 1 detects a little edge or stroke.

each of those neurons has 784 weights, one per pixel. reshape them back into a 28×28 image and you can *see* what pattern the neuron is looking for: bright = positive weight, dark = negative weight.

In [ ]:
weights = model.layer1.weight.detach()  # W for layer 1
print("weight matrix shape:", weights.shape)  # 16 neurons × 784 pixels

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    neuron_weights = weights[i].reshape(___, ___)  # hint: back into an image's shape
    ax.imshow(neuron_weights, cmap="gray")
    ax.set_title(f"neuron {i}")
    ax.axis("off")
plt.suptitle("what each hidden layer 1 neuron looks for")
plt.show()

**question:** do these look like edges and loops, or like noise? _write your answer here_

tip: to save the plot for your site, add `plt.savefig("neuron_weights.png")` before `plt.show()`, then put the file in `static/img/figures/`.

## step 6: break it
**chapter 2 claim:** show the network random noise and it still confidently says it's a digit.

`softmax` turns the 10 output scores into probabilities that add up to 1.

In [ ]:
noise = torch.rand(1, 1, 28, 28)  # random pixels between 0 and 1

with torch.no_grad():
    probabilities = torch.softmax(model(noise), dim=1)[0]

plt.imshow(noise[0, 0], cmap="gray")
plt.axis("off")
plt.show()
print(f"the network says: {probabilities.argmax().item()} with {probabilities.max().item():.0%} confidence")

**question:** is it confident? run the cell a few times. does it always pick the same digit? _write your answer here_

## what i found
_fill in, then copy this into a new note on the site:_
- test accuracy:
- hidden neurons look like:
- on random noise:
- what surprised me:
- open questions: